In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

candidates = pd.read_csv('../data/processed/candidates_clean.csv')
jobs       = pd.read_csv('../data/processed/jobs_clean.csv')

candidate_tfidf = sp.load_npz('../data/processed/candidate_tfidf.npz')
job_tfidf       = sp.load_npz('../data/processed/job_tfidf.npz')

# ══════════════════════════════════════════════════════════════════
# COSINE SIMILARITY (skills match)
# Shape: (num_candidates, num_jobs)
# ══════════════════════════════════════════════════════════════════
skill_sim = cosine_similarity(candidate_tfidf, job_tfidf)
print("Skill similarity matrix shape:", skill_sim.shape)

# ══════════════════════════════════════════════════════════════════
# EXPERIENCE LEVEL BOOST
# Add a small bonus when candidate level matches job level
# ══════════════════════════════════════════════════════════════════
def experience_bonus(cand_level, job_level):
    """Returns 0.10 bonus if levels match, 0.05 if one level apart, else 0."""
    diff = abs(int(cand_level) - int(job_level))
    if diff == 0:
        return 0.10
    elif diff == 1:
        return 0.05
    return 0.0

exp_bonus_matrix = np.zeros((len(candidates), len(jobs)))

for i, c_exp in enumerate(candidates['Exp_Numeric'].fillna(0)):
    for j, j_exp in enumerate(jobs['Exp_Numeric'].fillna(0)):
        exp_bonus_matrix[i, j] = experience_bonus(c_exp, j_exp)

# ══════════════════════════════════════════════════════════════════
# FINAL COMBINED SCORE
# 85% skill similarity + 15% experience bonus
# ══════════════════════════════════════════════════════════════════
final_score = 0.85 * skill_sim + 0.15 * exp_bonus_matrix

print("Final score matrix shape:", final_score.shape)
print("Score range: {:.4f} to {:.4f}".format(final_score.min(), final_score.max()))

# ══════════════════════════════════════════════════════════════════
# GENERATE TOP-N RECOMMENDATIONS PER CANDIDATE
# ══════════════════════════════════════════════════════════════════
TOP_N = 3

all_recommendations = []

for i, cand_row in candidates.iterrows():
    scores = final_score[i]
    # Get top N job indices (sorted descending)
    top_indices = np.argsort(scores)[::-1][:TOP_N]
    
    for rank, job_idx in enumerate(top_indices, start=1):
        job_row = jobs.iloc[job_idx]
        all_recommendations.append({
            'Candidate_Name':      cand_row['Name'],
            'Candidate_Skills':    cand_row['Skills_Clean'],
            'Candidate_Exp_Level': cand_row['Experience_Level'],
            'Rank':                rank,
            'Job_Title':           job_row['Job Title'],
            'Company':             job_row['Company'],
            'Location':            job_row['Location'],
            'Required_Skills':     job_row['Required_Skills_Clean'],
            'Job_Exp_Level':       job_row['Experience Level'],
            'Salary':              job_row['Salary'],
            'Industry':            job_row['Industry'],
            'Skill_Similarity':    round(skill_sim[i, job_idx], 4),
            'Final_Score':         round(final_score[i, job_idx], 4),
        })

recs_df = pd.DataFrame(all_recommendations)
print(f"\nTotal recommendation rows: {len(recs_df)}")
display(recs_df.head(12))

# Save
output_dir = Path('../outputs')
output_dir.mkdir(parents=True, exist_ok=True)
recs_df.to_csv(output_dir / 'recommendations.csv', index=False)
print("\nRecommendations saved to outputs/recommendations.csv")

# ══════════════════════════════════════════════════════════════════
# QUICK PREVIEW — show top 3 for each of the first 5 candidates
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("SAMPLE RECOMMENDATIONS")
print("="*70)

for name in candidates['Name'].dropna().head(5):
    subset = recs_df[recs_df['Candidate_Name'] == name].head(3)
    print(f"\n► {name} ({subset.iloc[0]['Candidate_Exp_Level']})")
    print(f"  Skills: {subset.iloc[0]['Candidate_Skills']}")
    for _, row in subset.iterrows():
        print(f"  #{row['Rank']}  {row['Job_Title']} @ {row['Company']} "
              f"[Score: {row['Final_Score']}]")

Skill similarity matrix shape: (815, 50000)
Final score matrix shape: (815, 50000)
Score range: 0.0000 to 0.4965

Total recommendation rows: 2445


,Candidate_Name,Candidate_Skills,Candidate_Exp_Level,Rank,Job_Title,Company,Location,Required_Skills,Job_Exp_Level,Salary,Industry,Skill_Similarity,Final_Score
0,Tara Gonzalez,"python, machine learning, numpy, scikit-learn,...",Entry Level,1,"Engineer, agricultural",Anderson-Bates,Sydney,"machine learning, python, sql",Entry Level,112000.0,Software,0.4640,0.4094
1,Tara Gonzalez,"python, machine learning, numpy, scikit-learn,...",Entry Level,2,"Programmer, applications",Miller Group,Bangalore,"machine learning, python, sql",Entry Level,50000.0,Software,0.4640,0.4094
2,Tara Gonzalez,"python, machine learning, numpy, scikit-learn,...",Entry Level,3,Diplomatic Services operational officer,Garrett Ltd,Toronto,"machine learning, sql, python",Entry Level,118000.0,Software,0.4640,0.4094
3,Jared Quinn,"numpy, scikit-learn, statistics, tensorflow, m...",Mid Level,1,Research scientist (physical sciences),Gonzalez Group,Bangalore,machine learning,Mid Level,93000.0,Software,0.2426,0.2212
4,Jared Quinn,"numpy, scikit-learn, statistics, tensorflow, m...",Mid Level,2,Chartered accountant,Perry Ltd,Berlin,machine learning,Mid Level,109000.0,Software,0.2426,0.2212
5,Jared Quinn,"numpy, scikit-learn, statistics, tensorflow, m...",Mid Level,3,Adult guidance worker,Howell-Ellis,Berlin,machine learning,Mid Level,86000.0,Software,0.2426,0.2212
6,Jane Woods,"tensorflow, python, numpy, machine learning, nlp",Mid Level,1,Analytical chemist,Clarke and Sons,San Francisco,"machine learning, python",Mid Level,56000.0,Software,0.3457,0.3088
7,Jane Woods,"tensorflow, python, numpy, machine learning, nlp",Mid Level,2,Human resources officer,Sullivan Inc,London,"python, machine learning",Mid Level,102000.0,Software,0.3457,0.3088
8,Jane Woods,"tensorflow, python, numpy, machine learning, nlp",Mid Level,3,Estate manager/land agent,Hayes LLC,Toronto,"python, machine learning",Mid Level,93000.0,Software,0.3457,0.3088
9,Julie Chambers,"scikit-learn, statistics, data visualization, ...",Entry Level,1,Museum/gallery curator,Meyer and Sons,Sydney,sql,Entry Level,46000.0,Software,0.1914,0.1777



Recommendations saved to outputs/recommendations.csv

SAMPLE RECOMMENDATIONS

► Tara Gonzalez (Entry Level)
  Skills: python, machine learning, numpy, scikit-learn, sql
  #1  Engineer, agricultural @ Anderson-Bates [Score: 0.4094]
  #2  Programmer, applications @ Miller Group [Score: 0.4094]
  #3  Diplomatic Services operational officer @ Garrett Ltd [Score: 0.4094]

► Jared Quinn (Mid Level)
  Skills: numpy, scikit-learn, statistics, tensorflow, machine learning
  #1  Research scientist (physical sciences) @ Gonzalez Group [Score: 0.2212]
  #2  Chartered accountant @ Perry Ltd [Score: 0.2212]
  #3  Adult guidance worker @ Howell-Ellis [Score: 0.2212]

► Jane Woods (Mid Level)
  Skills: tensorflow, python, numpy, machine learning, nlp
  #1  Analytical chemist @ Clarke and Sons [Score: 0.3088]
  #2  Human resources officer @ Sullivan Inc [Score: 0.3088]
  #3  Estate manager/land agent @ Hayes LLC [Score: 0.3088]

► Julie Chambers (Entry Level)
  Skills: scikit-learn, statistics, data v